# Distinct Rows — Removing Duplicate Rows

Remove duplicate records across the intermediate DataFrame with full control over column subsets.

`pytae` provides both direct method chaining `df.distinct(*cols, keep='first')` and the functional verb `pt.distinct(df, *cols)`. In the CLI, the equivalent flag is `-distinct` (or `-distinct COLS`).

| Operation | Pytae Method Chain | CLI Equivalent |
|---|---|---|
| All Columns | `df.distinct()` | `-distinct` |
| Specific Columns | `df.distinct('col1', 'col2')` | `-distinct "col1,col2"` |
| Keep Last / None | `df.distinct(keep='last')` | N/A |

---

In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt
import pandas as pd

penguins = pt.sample_data['penguins']
tips = pt.sample_data['tips']

## 1. Deduplicating Across All Columns (`.distinct()`)

Eliminate identical rows across all selected columns. The index is automatically reset:

In [2]:
# Keep only unique species and island pairs
(
    penguins
    .select("species", "island")
    .distinct()
)

species,island
str,str
"""Adelie""","""Torgersen"""
"""Adelie""","""Biscoe"""
"""Adelie""","""Dream"""
"""Chinstrap""","""Dream"""
"""Gentoo""","""Biscoe"""


## 2. Deduplicating by Specific Identifier Columns

Pass column names to restrict duplicate checks to specific identifiers, keeping the first matching record for all other columns:

In [3]:
# Keep the first recorded penguin for each unique species and island combination
(
    penguins
    .select("species", "island", "body_mass_g", "sex")
    .distinct("species", "island")
)

species,island,body_mass_g,sex
str,str,f64,str
"""Adelie""","""Torgersen""",3750.0,"""Male"""
"""Adelie""","""Biscoe""",3400.0,"""Female"""
"""Adelie""","""Dream""",3250.0,"""Female"""
"""Chinstrap""","""Dream""",3500.0,"""Female"""
"""Gentoo""","""Biscoe""",4500.0,"""Female"""


## 3. Chaining in Pipelines

`df.distinct()` flows seamlessly inside multi-step pipelines in fluent method chains:

In [4]:
# Filter heavy penguins -> select columns -> deduplicate -> arrange
(
    penguins
    .filter("body_mass_g > 4000")
    .select("species", "island")
    .distinct()
    .arrange("species", "island")
)

species,island
str,str
"""Adelie""","""Biscoe"""
"""Adelie""","""Dream"""
"""Adelie""","""Torgersen"""
"""Chinstrap""","""Dream"""
"""Gentoo""","""Biscoe"""


## 4. Keeping the Last Record (`keep='last'`)

Optionally retain the last occurrence instead of the first:

In [5]:
# Retain the last record for each day and time
(
    tips
    .select("day", "time", "total_bill", "tip")
    .distinct("day", "time", keep="last")
)

day,time,total_bill,tip
cat,cat,f64,f64
"""Fri""","""Dinner""",15.38,3.0
"""Sun""","""Dinner""",15.69,1.5
"""Thur""","""Lunch""",16.47,3.23
"""Fri""","""Lunch""",10.09,2.0
"""Sat""","""Dinner""",17.82,1.75
"""Thur""","""Dinner""",18.78,3.0


## 5. Functional Style (`pt.distinct`)

Available as a standalone function:

In [6]:
# Functional deduplication
pt.distinct(penguins[["species", "island"]])

species,island
str,str
"""Adelie""","""Torgersen"""
"""Adelie""","""Biscoe"""
"""Adelie""","""Dream"""
"""Chinstrap""","""Dream"""
"""Gentoo""","""Biscoe"""
